# Coriolis Pressure Correction Example (CMF300H)

This notebook demonstrates pressure correction of Coriolis mass flow for a real-life calibration case at an accredited calibration facility. 

## Case
- Meter: **Emerson Micro Motion CMF300H**
- Calibration pressure: **60 bara**
- Meter Factory calibration pressure (`P_cal`): **2.55 bara**
- Reference accumulated mass: **4120.487 kg**
- Coriolis accumulated mass (pressure compensation disabled): **4105.007 kg**
- Flow rate during run: **82 tonnes/hr**

The CMF300H process-pressure effect from PDS is **-0.006 % of rate per bar**.
The equation input `PCm` is the **correction factor**, i.e. opposite sign of PDS effect:

$$PCm = -\,(\text{PDS process-pressure effect}) = +0.006\;\%/\text{bar}$$

## Equations

Mass-flow pressure correction:

$$m_{corr} = m_{meas}\left(1 + \frac{PCm}{100}(P_{act}-P_{cal})\right)$$

Error relative to reference meter:

$$\text{error}~[\%] = 100\cdot\frac{m - m_{ref}}{m_{ref}}$$

In [1]:
import pandas as pd
from pvtlib.metering import coriolis_flowmeters

In [2]:
# Input data
p_act_bara = 60.0
p_cal_bara = 2.55
reference_mass_kg = 4120.487
measured_mass_kg = 4105.007
flow_rate_tonnes_per_hr = 82.0

# CMF300H process-pressure effect from PDS [%/bar]
pds_process_pressure_effect_pct_per_bar = -0.006

# Equation input PCm is opposite sign of PDS process-pressure effect
pc_m_pct_per_bar = -pds_process_pressure_effect_pct_per_bar

In [3]:
corrected_mass_kg = coriolis_flowmeters.coriolis_massflow_corr_pres(
    m_meas=measured_mass_kg,
    P_act=p_act_bara,
    PCm=pc_m_pct_per_bar,
    P_cal=p_cal_bara,
)

error_without_correction_pct = 100.0 * (measured_mass_kg - reference_mass_kg) / reference_mass_kg
error_with_correction_pct = 100.0 * (corrected_mass_kg - reference_mass_kg) / reference_mass_kg

In [4]:
results = pd.DataFrame(
    [
        {
            "Condition": "Without pressure correction",
            "Accumulated mass [kg]": measured_mass_kg,
            "Error [%]": error_without_correction_pct,
        },
        {
            "Condition": "With pressure correction",
            "Accumulated mass [kg]": corrected_mass_kg,
            "Error [%]": error_with_correction_pct,
        },
    ]
)

results.round({"Accumulated mass [kg]": 3, "Error [%]": 4})

,Condition,Accumulated mass [kg],Error [%]
0,Without pressure correction,4105.007,-0.3757
1,With pressure correction,4119.157,-0.0323


This concludes that the original deviation is explained by the process pressure effect. This is the expected behavior of a Coriolis flowmeter operating at a different pressure than the pressure at which it was originally calibrated.